# MAX Experimental Tensor API

> Research notebook for async experimental tensor API.
> This API uses lazy evaluation - operations don't execute until `.realize` is called.

**Status**: Research - not integrated into mxframe yet.

**Issue**: The experimental API uses `asyncio.run()` internally when you try to get values synchronously,
which conflicts with Jupyter's event loop or other async contexts.

In [ ]:
import numpy as np
import asyncio
from max.experimental import tensor as mx_tensor
from max import driver

## Basic Usage - Async Approach

The experimental tensor API is designed for async workflows.
Operations are lazy until you call `await tensor.realize`.

In [ ]:
# Create test data
arr = np.array([1.0, 2.0, 3.0, 4.0, 5.0], dtype=np.float32)
print(f"Input: {arr}")

In [ ]:
# Async function to use experimental tensor
async def experimental_sum(arr: np.ndarray) -> float:
    """Sum using experimental tensor API with proper async."""
    t = mx_tensor.from_numpy(arr)
    result = t.sum()
    await result.realize  # Proper async materialization
    return float(np.from_dlpack(result))


async def experimental_mean(arr: np.ndarray) -> float:
    """Mean using experimental tensor API."""
    t = mx_tensor.from_numpy(arr)
    result = t.mean()
    await result.realize
    return float(np.from_dlpack(result))


async def experimental_add(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Element-wise addition using experimental tensor API."""
    ta = mx_tensor.from_numpy(a)
    tb = mx_tensor.from_numpy(b)
    result = ta + tb
    await result.realize
    return np.from_dlpack(result)


async def experimental_mul(a: np.ndarray, scalar: float) -> np.ndarray:
    """Scalar multiplication using experimental tensor API."""
    ta = mx_tensor.from_numpy(a)
    # Note: Check if scalar multiplication works directly
    tb = mx_tensor.from_numpy(np.full(a.shape, scalar, dtype=a.dtype))
    result = ta * tb
    await result.realize
    return np.from_dlpack(result)

## Running in Jupyter

In Jupyter, you can use `await` directly in cells since Jupyter runs an event loop.

In [ ]:
# This should work in Jupyter because Jupyter has its own event loop
# await experimental_sum(arr)

## Sync Wrapper Approach

If we need to call from sync code, we need to handle the event loop properly.
This is where it gets tricky - `asyncio.run()` fails if there's already an event loop.

In [ ]:
def sync_wrapper(coro):
    """
    Run async coroutine from sync context.
    Handles the case where an event loop is already running (e.g., Jupyter).
    """
    try:
        loop = asyncio.get_running_loop()
        # Already in an event loop - use nest_asyncio or create task
        import nest_asyncio
        nest_asyncio.apply()
        return asyncio.run(coro)
    except RuntimeError:
        # No event loop running - safe to use asyncio.run()
        return asyncio.run(coro)


# Alternative: Use asyncio.get_event_loop().run_until_complete()
def sync_wrapper_v2(coro):
    """
    Alternative sync wrapper.
    """
    try:
        loop = asyncio.get_running_loop()
        # Schedule and wait - but this doesn't work directly
        # Need to use concurrent.futures or similar
        import concurrent.futures
        with concurrent.futures.ThreadPoolExecutor() as pool:
            future = pool.submit(asyncio.run, coro)
            return future.result()
    except RuntimeError:
        return asyncio.run(coro)

## GPU Support Check

The experimental tensor API should support GPU via device specification.

In [ ]:
# Check if GPU is available
print(f"GPU count: {driver.accelerator_count()}")

# TODO: Check how to specify device in experimental tensor API
# Possible options:
# - mx_tensor.from_numpy(arr, device="gpu")
# - tensor.to(device)
# - Context manager for device

## Potential Solutions to Research

1. **nest_asyncio**: `pip install nest_asyncio` - allows nested event loops
2. **Run in thread pool**: Execute async code in a separate thread
3. **Make all mxframe async**: Change API to be async-first
4. **Check Modular forums**: See if there's an official sync API

### Questions for Modular Forums:

1. Is there a sync version of the experimental tensor API?
2. What's the recommended way to use experimental tensors from sync code?
3. How do you specify GPU device with experimental tensors?
4. Is `nest_asyncio` safe to use with MAX Engine?

In [ ]:
# Test with nest_asyncio (need to pip install nest_asyncio first)
# import nest_asyncio
# nest_asyncio.apply()
# 
# result = asyncio.run(experimental_sum(arr))
# print(f"Sum: {result}")

## Available Experimental Tensor Operations

From the API docs, these methods are available on `Tensor`:

- `.sum()` - Reduction
- `.mean()` - Reduction  
- `.max()` - Reduction
- `+`, `-`, `*`, `/` - Arithmetic operators
- Comparison operators
- `.reshape()`, `.transpose()`
- `.realize` - Async materialization

In [ ]:
# List available methods on experimental tensor
# t = mx_tensor.from_numpy(arr)
# print([m for m in dir(t) if not m.startswith('_')])